# DiffuLLaMA: restored paper experiments

This notebook launches the package implementations for all ten corrected experiments. It does not duplicate scientific code. Expensive cells are opt-in and results persist in Google Drive.

## Recommended execution order

1. Setup, authentication, and Drive
2. Install dependencies
3. CPU tests
4. Prepare EWT, German GSD, and Japanese GSD
5. Dry-run smoke test
6. Real-model smoke test
7. Relation-Head Receiver Prediction
8. Validate the six selection locks
9. Relation-Head Receiver Prediction over Diffusion Time
10. Attention Entropy
11. POS/Token-Class Linear Probes
12. Final-Token Prediction by Layer
13. Prediction Before Unmasking: Timing Analysis
14. Direct Logit Attribution
15. Matched Relation-Head Ablation
16. Attention Heatmaps and Trajectories
17. German/Japanese Multilingual Relation-Head Transfer
18. Validate outputs
19. Package summaries and figures

## Runtime and storage warning

A 7B model plus 64-step trajectories is substantial GPU work. Expect many GPU-hours and tens to hundreds of GB for checkpoints, native histories, attention evidence, and figures. Run one experiment at a time. Do not enable the costly cells until the CPU tests and both smoke tests pass.

In [ ]:
import subprocess

subprocess.run(["nvidia-smi"], check=False)

## Setup, secrets, exact checkout, and Drive

Create `GH_TOKEN` and `HF_TOKEN` in Colab Secrets. They are read without being printed. Edit `GIT_COMMIT` only when deliberately moving to another reviewed commit.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

from google.colab import drive, userdata

GIT_COMMIT = "8a56e00b1dbec4081caf1f288fec02d8da2dd600"
WORK_ROOT = Path(os.environ.get("COLAB_WORK_ROOT", "/content"))
REPOSITORY = WORK_ROOT / "latentrelationsondlm"
MODEL_CONFIG = "configs/models/diffullama_7b.yaml"
MODEL_ID = "diffullama_7b"
RESULT_ROOT = WORK_ROOT / "drive" / "MyDrive" / "dlmrel-paper-results" / "diffullama"
RUN_PREFIX = "paper-restoration-v1-diffullama"
RUN_REAL_SMOKE = False
RUN_EXPENSIVE = False

gh_token = userdata.get("GH_TOKEN")
hf_token = userdata.get("HF_TOKEN")
if not gh_token or not hf_token:
    raise RuntimeError("Add GH_TOKEN and HF_TOKEN to Colab Secrets before continuing.")
os.environ["HF_TOKEN"] = hf_token
subprocess.run(
    ["gh", "auth", "login", "--with-token"],
    input=gh_token + "\n",
    text=True,
    check=True,
    capture_output=True,
)
del gh_token, hf_token

drive.mount(str(WORK_ROOT / "drive"))
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["DLMREL_STANFORD_POS_CACHE"] = str(
    WORK_ROOT / "drive" / "MyDrive" / "dlmrel-paper-dependencies" / "stanford-pos-4.2.0"
)
if not REPOSITORY.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/Dabsoysauce/latentrelationsondlm.git", str(REPOSITORY)],
        check=True,
    )
subprocess.run(["git", "-C", str(REPOSITORY), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(REPOSITORY), "checkout", "--detach", GIT_COMMIT], check=True)
actual_sha = subprocess.check_output(["git", "-C", str(REPOSITORY), "rev-parse", "HEAD"], text=True).strip()
print("Checked out:", actual_sha)
if actual_sha != GIT_COMMIT:
    raise RuntimeError("Checked-out SHA does not match GIT_COMMIT")
os.chdir(REPOSITORY)

## Install the project and model-specific environment

In [ ]:
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[dev]", "-r", "requirements/diffullama.txt"],
    check=True,
)

## Complete CPU test suite

Do not continue to GPU cells unless this passes.

In [ ]:
import sys

subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
subprocess.run([sys.executable, "-m", "ruff", "check", "."], check=True)

## Prepare frozen manifests

This preserves the official files. Corrected runners consume selection/test only and never open development data.

In [ ]:
for dataset in (
    "configs/datasets/ewt.yaml",
    "configs/datasets/de_gsd.yaml",
    "configs/datasets/ja_gsd.yaml",
):
    subprocess.run([sys.executable, "-m", "dlmrel.cli", "prepare", "--dataset", dataset], check=True)

## Restart-safe launch helpers

A completed run is skipped. Incomplete runs use `--resume`. Selection-lock consumers receive the exact lock directory.

In [ ]:
EXPERIMENT_TRACK = {
    "relation_head_receiver_prediction": "confirmatory_ewt",
    "relation_head_receiver_prediction_over_diffusion_time": "confirmatory_ewt",
    "attention_entropy": "exploratory_extensions",
    "pos_token_class_linear_probes": "exploratory_extensions",
    "final_token_prediction_by_layer": "exploratory_extensions",
    "prediction_before_unmasking_timing_analysis": "exploratory_extensions",
    "direct_logit_attribution": "exploratory_extensions",
    "matched_relation_head_ablation": "exploratory_extensions",
    "attention_heatmaps_and_trajectories": "exploratory_extensions",
    "multilingual_relation_head_transfer": "external_treebank_transfer",
}
DATASET_ID = {"ewt.yaml": "ewt", "de_gsd.yaml": "de_gsd", "ja_gsd.yaml": "ja_gsd"}

def require_expensive():
    if not RUN_EXPENSIVE:
        raise RuntimeError("Set RUN_EXPENSIVE = True only when you intend to spend GPU time.")

def expected_run_dir(experiment, dataset, run_id):
    return (
        RESULT_ROOT / EXPERIMENT_TRACK[experiment] / MODEL_ID
        / DATASET_ID[Path(dataset).name] / experiment / run_id
    )

def run_experiment(
    experiment, run_id, dataset="configs/datasets/ewt.yaml", selection_lock=None,
    export_attention_cache=False, attention_cache=None,
):
    require_expensive()
    target = expected_run_dir(experiment, dataset, run_id)
    summary = target / "summary.json"
    if summary.is_file() and json.loads(summary.read_text()).get("completion_status") == "complete":
        print("Already complete; skipping:", target)
        return target
    command = [
        sys.executable, "-m", "dlmrel.cli", "run",
        "--model", MODEL_CONFIG,
        "--dataset", dataset,
        "--experiment", f"configs/experiments/{experiment}.yaml",
        "--results", str(RESULT_ROOT),
        "--run-id", run_id,
        "--resume",
        "--timestep-batch-size", "8",
    ]
    if selection_lock is not None:
        command.extend(["--selection-lock", str(selection_lock)])
    if export_attention_cache:
        command.append("--export-attention-cache")
    if attention_cache is not None:
        command.extend(["--attention-cache", str(attention_cache)])
    subprocess.run(command, check=True)
    return target

SELECTION_RUN_ID = f"{RUN_PREFIX}-relation-selection"
SELECTION_RUN = expected_run_dir(
    "relation_head_receiver_prediction",
    "configs/datasets/ewt.yaml",
    SELECTION_RUN_ID,
)
LOCK_DIR = SELECTION_RUN / "selection-locks"
TIME_RUN_ID = f"{RUN_PREFIX}-relation-head-receiver-prediction-over-diffusion-time"
TIME_RUN = expected_run_dir(
    "relation_head_receiver_prediction_over_diffusion_time",
    "configs/datasets/ewt.yaml",
    TIME_RUN_ID,
)
POS_RUN_ID = f"{RUN_PREFIX}-pos-token-class-linear-probes"
POS_RUN = expected_run_dir(
    "pos_token_class_linear_probes", "configs/datasets/ewt.yaml", POS_RUN_ID
)
if (POS_RUN / "pos_head_rankings.csv").is_file():
    os.environ["DLMREL_POS_HEAD_RANKINGS"] = str(POS_RUN)

## Dry-run smoke test

This validates configuration without loading the 7B model.

In [ ]:
subprocess.run([
    sys.executable, "-m", "dlmrel.cli", "smoke-test",
    "--model", MODEL_CONFIG,
    "--dataset", "configs/datasets/ewt.yaml",
    "--experiment", "configs/experiments/relation_head_receiver_prediction.yaml",
    "--dry-run",
], check=True)

## Real-model smoke test

This is deliberately separate from the dry run and opt-in.

In [ ]:
if not RUN_REAL_SMOKE:
    raise RuntimeError("Set RUN_REAL_SMOKE = True only when the GPU runtime is ready.")
subprocess.run([
    sys.executable, "-m", "dlmrel.cli", "smoke-test",
    "--model", MODEL_CONFIG,
    "--dataset", "configs/datasets/ewt.yaml",
    "--experiment", "configs/experiments/relation_head_receiver_prediction.yaml",
], check=True)

## Relation-Head Receiver Prediction

Run this first. It scores fully visible EWT selection sentences, publishes six model-specific locks, and only then opens held-out test data.

In [ ]:
SELECTION_RUN = run_experiment(
    "relation_head_receiver_prediction",
    SELECTION_RUN_ID,
)
LOCK_DIR = SELECTION_RUN / "selection-locks"
print("Selection locks:", LOCK_DIR)

## Validate all six selection locks

Dependent experiments must not run until this command reports all six relations for this exact model revision.

In [ ]:
subprocess.run([
    sys.executable, "-m", "dlmrel.cli", "validate-selection-locks",
    "--model", MODEL_CONFIG,
    "--dataset", "configs/datasets/ewt.yaml",
    "--experiment", "configs/experiments/relation_head_receiver_prediction.yaml",
    "--selection-lock", str(LOCK_DIR),
], check=True)

## Relation-Head Receiver Prediction over Diffusion Time

Depends on the validated English selection locks.

In [ ]:
TIME_RUN = run_experiment(
    "relation_head_receiver_prediction_over_diffusion_time",
    TIME_RUN_ID,
    selection_lock=LOCK_DIR,
    export_attention_cache=True,
)
print(TIME_RUN)

## Attention Entropy

This cell reuses the validated entropy cache written by the English relation-time run, so it performs no second 64-step GPU sweep.

In [ ]:
RUN = run_experiment(
    "attention_entropy",
    f"{RUN_PREFIX}-attention-entropy",
    attention_cache=TIME_RUN,
)
print(RUN)

## POS/Token-Class Linear Probes

The runner automatically downloads checksum-pinned Stanford tagger 4.2.0 to the persistent Drive dependency cache and reuses it after restarts. It never substitutes UD UPOS. The exact historical Stanford release remains an explicit provenance limitation.

In [ ]:
POS_RUN = run_experiment(
    "pos_token_class_linear_probes",
    POS_RUN_ID,
)
os.environ["DLMREL_POS_HEAD_RANKINGS"] = str(POS_RUN)
print(POS_RUN)

## Final-Token Prediction by Layer

This cell has no selection-lock dependency.

In [ ]:
RUN = run_experiment(
    "final_token_prediction_by_layer",
    f"{RUN_PREFIX}-final-token-prediction-by-layer",
)
print(RUN)

## Prediction Before Unmasking: Timing Analysis

This cell has no selection-lock dependency.

In [ ]:
RUN = run_experiment(
    "prediction_before_unmasking_timing_analysis",
    f"{RUN_PREFIX}-prediction-before-unmasking-timing-analysis",
)
print(RUN)

## Direct Logit Attribution

Depends on the validated English selection locks.

In [ ]:
RUN = run_experiment(
    "direct_logit_attribution",
    f"{RUN_PREFIX}-direct-logit-attribution",
    selection_lock=LOCK_DIR,
)
print(RUN)

## Matched Relation-Head Ablation

Depends on selection locks. When the POS cell has completed, its most-decodable versus lower-decoding head comparison is included automatically, including after a runtime restart.

In [ ]:
RUN = run_experiment(
    "matched_relation_head_ablation",
    f"{RUN_PREFIX}-matched-relation-head-ablation",
    selection_lock=LOCK_DIR,
)
print(RUN)

## Attention Heatmaps and Trajectories

Depends on the validated English selection locks.

In [ ]:
RUN = run_experiment(
    "attention_heatmaps_and_trajectories",
    f"{RUN_PREFIX}-attention-heatmaps-and-trajectories",
    selection_lock=LOCK_DIR,
)
print(RUN)

## Multilingual Relation-Head Transfer: German GSD

Uses this model's frozen English locks and never reselects in German.

In [ ]:
GERMAN_RUN = run_experiment(
    "multilingual_relation_head_transfer",
    f"{RUN_PREFIX}-multilingual-transfer-de-gsd",
    dataset="configs/datasets/de_gsd.yaml",
    selection_lock=LOCK_DIR,
)
print(GERMAN_RUN)

## Multilingual Relation-Head Transfer: Japanese GSD

Uses this model's frozen English locks and never reselects in Japanese.

In [ ]:
JAPANESE_RUN = run_experiment(
    "multilingual_relation_head_transfer",
    f"{RUN_PREFIX}-multilingual-transfer-ja-gsd",
    dataset="configs/datasets/ja_gsd.yaml",
    selection_lock=LOCK_DIR,
)
print(JAPANESE_RUN)

## Inspect compact summaries

This reads `summary.json` and a small CSV preview, not enormous Parquet evidence.

In [ ]:
for summary_path in sorted(RESULT_ROOT.rglob("summary.json")):
    run_dir = summary_path.parent
    subprocess.run([
        sys.executable, "-m", "dlmrel.cli", "summarize",
        "--run-dir", str(run_dir), "--rows", "5",
    ], check=True)

## Validate every completed output

In [ ]:
for summary_path in sorted(RESULT_ROOT.rglob("summary.json")):
    subprocess.run([
        sys.executable, "-m", "dlmrel.cli", "validate",
        "--run-dir", str(summary_path.parent),
    ], check=True)

## Package summaries and figures

Large Parquet and checkpoint files remain in Drive; this compact archive is convenient for review.

In [ ]:
import shutil

PACKAGE_ROOT = RESULT_ROOT / "review-package"
PACKAGE_ROOT.mkdir(exist_ok=True)
for source in RESULT_ROOT.rglob("*"):
    is_review_artifact = (
        source.name in {"summary.json", "metrics.csv", "per_seed_metrics.csv"}
        or source.suffix == ".pdf"
    )
    if source.is_file() and is_review_artifact:
        relative = source.relative_to(RESULT_ROOT)
        destination = PACKAGE_ROOT / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
archive = shutil.make_archive(str(RESULT_ROOT / f"{RUN_PREFIX}-summaries-and-figures"), "zip", PACKAGE_ROOT)
print("Wrote:", archive)